# A swing beside a vanilla call

`call_swing` is not a call, and the gap is large enough to matter: on the numbers below the
model's default swing is worth **less than half** the equivalent right. This notebook prices
them side by side so the difference is attributable rather than surprising.

It works in three steps:

1. **Fix the vol convention.** The model mean-reverts, so terminal variance is not `sigma^2 T`.
   Comparing at the raw `sVol` makes the model look 25 % cheap when it is correct.
2. **Anchor the model on a closed form.** Collapse the swing to one exercise day with the
   quota optional and it *is* a European call. It should reproduce Black-76.
3. **Walk the ladder** from that call up to the contract the model actually prices, one
   difference at a time: timing, multiple exercises, then obligation.

Conventions are in [docs/MODEL-CONVENTIONS.md](docs/MODEL-CONVENTIONS.md). Everything here is
undiscounted by default — see §1.

In [ ]:
import os, sys, warnings
from math import erf, exp, log, sqrt

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import storage_model as sm

pd.set_option("display.width", 200, "display.max_columns", 50)
plt.rcParams.update({"figure.figsize": (12, 3.4), "axes.grid": True, "grid.alpha": .25,
                     "axes.spines.top": False, "axes.spines.right": False, "font.size": 9})
warnings.filterwarnings("ignore", category=FutureWarning)

# `import storage_model` is cached, so a kernel older than the symbols used here
# would fail part way down rather than at the top. Fail at the top instead.
_missing = [n for n in ("Storage", "run_valuation") if not hasattr(sm, n)]
if _missing:
    raise RuntimeError(
        "This kernel is running an older copy of storage_model — missing "
        + ", ".join(_missing) + ". Restart the kernel (Kernel > Restart Kernel and Run All).")

SMOKE = os.environ.get("STORAGE_NOTEBOOK_SMOKE") == "1"
print(f"ready — storage_model from {sm.__file__}")

## 1. Inputs

One flat forward curve, so nothing here depends on curve shape and every difference between
the contracts is structural. `DISCOUNT_RATE` is **0** on purpose: both legs of every
comparison would carry the same discount factor, so it cancels and only obscures the
structural gap. Time value is the subject of `Products.ipynb`, not this notebook.

In [ ]:
F_LEVEL       = 40.0                        # flat forward, EUR/MWh
STRIKE        = 40.0                        # at the money
VOL           = 0.50                        # annualised instantaneous log-vol (sVol)
SMR           = 1.0                         # mean reversion per year
DISCOUNT_RATE = 0.0                         # see the note above
VAL_DATE      = pd.Timestamp("2026-01-01")
WIN_START, WIN_END = "2027-01-01", "2027-12-31"
ONE_DATE      = pd.Timestamp("2027-07-01")  # the single-expiry benchmark, mid-window
MWH_PER_DAY   = 1_000
N_P           = 40 if SMOKE else 60         # tree half-width for the multi-day runs
N_P_ANCHOR    = 60 if SMOKE else 90         # wider, for the single-date closed-form check
DAY_COUNTS    = [10] if SMOKE else [1, 10, 90]

_span = pd.date_range("2025-01-01", "2029-12-31", freq="D")
CURVE = pd.Series(F_LEVEL, index=_span)

WINDOW_DAYS = (pd.Timestamp(WIN_END) - pd.Timestamp(WIN_START)).days + 1
T_ONE = (ONE_DATE - VAL_DATE).days / 365.25
print(f"flat {F_LEVEL:.2f} EUR/MWh, strike {STRIKE:.2f}, sVol {VOL}, sMR {SMR}, "
      f"discount {DISCOUNT_RATE:.0%}")
print(f"window {WIN_START} .. {WIN_END} = {WINDOW_DAYS} days;  single expiry {ONE_DATE:%Y-%m-%d} "
      f"is T = {T_ONE:.4f} y from {VAL_DATE:%Y-%m-%d}")

## 2. The vol convention

This is the step that goes wrong first. Under the model's Ornstein–Uhlenbeck log price the
terminal variance is **not** `sigma^2 T`. Clewlow & Strickland (1999a) give it for the
one-factor Schwartz model as the integral of forward return variance over the life of the
option — their equation (6.13), for an option expiring at `T` on a futures contract
delivering at `s`:

$$w^2 \;=\; \int_t^T \sigma^2 e^{-2lpha(s-u)}\,du \;=\; rac{\sigma^2}{2lpha}\left(e^{-2lpha(s-T)} - e^{-2lpha(s-t)}ight)$$

Two things follow, and only the first is used below.

**Spot options — the `s = T` case, their (6.14).** A swing exercises against the daily index
on the exercise day, so `s = T` and the expression collapses to

$$w^2 \;=\; rac{\sigma^2}{2lpha}\left(1 - e^{-2lpha(T-t)}ight)$$

which is what `ou_var()` returns. Feed the raw `sVol` into a vanilla formula instead and the
model looks cheap when it is right.

**Futures options — the Samuelson effect.** For `s > T` the variance is damped by
`exp(-2*alpha*(s-T))`, so the forward's volatility is `sigma * exp(-alpha*(s-t))`: a contract
delivering further out moves less for the same shock to spot. At `sMR = 1.0` a future
delivering a year after the option expires carries only 37 % of spot vol. **This model cannot
price that option** — it has no tradable futures with their own dynamics; exercise is
physical against the index. It is tabulated below only so the distinction is explicit.

In [ ]:
def ou_var(T, vol=None, mr=None):
    """Clewlow & Strickland (6.14): terminal log-variance of the spot at horizon T."""
    vol = VOL if vol is None else vol
    mr = SMR if mr is None else mr
    if mr <= 0:
        return vol ** 2 * T
    return vol ** 2 * (1.0 - exp(-2.0 * mr * T)) / (2.0 * mr)


def futures_var(T, s, vol=None, mr=None):
    """C&S (6.13): option expiring at T on a future delivering at s >= T, from t = 0."""
    vol = VOL if vol is None else vol
    mr = SMR if mr is None else mr
    if mr <= 0:
        return vol ** 2 * T
    return vol ** 2 / (2.0 * mr) * (exp(-2.0 * mr * (s - T)) - exp(-2.0 * mr * s))


def black76(fwd, strike, var, df=1.0):
    """Black-76 call on a forward, given TOTAL log-variance rather than a vol."""
    if var <= 0.0:
        return df * max(fwd - strike, 0.0)
    norm = lambda z: 0.5 * (1.0 + erf(z / sqrt(2.0)))
    sd = sqrt(var)
    d1 = (log(fwd / strike) + 0.5 * var) / sd
    return df * (fwd * norm(d1) - strike * norm(d1 - sd))


assert abs(futures_var(T_ONE, T_ONE) - ou_var(T_ONE)) < 1e-15, "(6.13) must collapse to (6.14)"

_T = np.array([0.25, 0.5, 1.0, T_ONE, 2.0, 5.0])
display(pd.DataFrame({
    "T (years)": _T,
    "total variance w^2": [ou_var(t) for t in _T],
    "effective Black-76 vol": [sqrt(ou_var(t) / t) for t in _T],
    "naive sVol": VOL,
    "error if you use sVol": [sqrt(ou_var(t) / t) / VOL - 1.0 for t in _T]})
    .style.hide(axis="index").format({
        "T (years)": "{:.3f}", "total variance w^2": "{:.5f}",
        "effective Black-76 vol": "{:.4f}", "naive sVol": "{:.4f}",
        "error if you use sVol": "{:+.1%}"})
    .set_caption(f"Spot options, C&S (6.14) — sVol {VOL}, sMR {SMR}. Variance saturates at "
                 f"sVol²/(2·sMR) = {VOL**2/(2*SMR):.4f}, so the comparable vol keeps falling"))

_grid = np.linspace(0.05, 5.0, 200)
fig, ax = plt.subplots(1, 2, figsize=(12, 3.4))
ax[0].plot(_grid, [ou_var(t) for t in _grid], color="black", lw=1.6, label="w² (mean reverting)")
ax[0].plot(_grid, [VOL ** 2 * t for t in _grid], color="tab:blue", ls="--", lw=1,
           label="sVol²·T (what a GBM would give)")
ax[0].axhline(VOL ** 2 / (2 * SMR), color="tab:red", ls=":", lw=1,
              label=f"saturates at sVol²/(2·sMR) = {VOL**2/(2*SMR):.4f}")
ax[0].set_ylim(0, 0.35); ax[0].set_xlabel("years to expiry"); ax[0].set_ylabel("total variance")
ax[0].set_title("Variance saturates — it does not grow with T"); ax[0].legend(fontsize=7)

ax[1].plot(_grid, [sqrt(ou_var(t) / t) for t in _grid], color="black", lw=1.6,
           label="effective Black-76 vol")
ax[1].axhline(VOL, color="tab:blue", ls="--", lw=1, label=f"sVol = {VOL} (the T → 0 limit)")
ax[1].axvline(T_ONE, color="grey", lw=.8)
ax[1].set_xlabel("years to expiry"); ax[1].set_ylabel("annualised vol")
ax[1].set_title("so the comparable vol falls towards zero"); ax[1].legend(fontsize=7)
plt.tight_layout(); plt.show()

# The general case, for reference only: this model prices spot exercise, not futures options.
_s = np.array([0.0, 0.25, 0.5, 1.0, 2.0])
display(pd.DataFrame({
    "s - T (years to delivery, after expiry)": _s,
    "w^2, C&S (6.13)": [futures_var(1.0, 1.0 + d) for d in _s],
    "implied vol": [sqrt(futures_var(1.0, 1.0 + d) / 1.0) for d in _s],
    "share of spot vol": [sqrt(futures_var(1.0, 1.0 + d) / futures_var(1.0, 1.0)) for d in _s],
    "exp(-sMR·(s-T))": [exp(-SMR * d) for d in _s]})
    .style.hide(axis="index").format({
        "s - T (years to delivery, after expiry)": "{:.2f}", "w^2, C&S (6.13)": "{:.6f}",
        "implied vol": "{:.4f}", "share of spot vol": "{:.3f}", "exp(-sMR·(s-T))": "{:.3f}"})
    .set_caption("Samuelson effect — a 1-year option on a future delivering at s. "
                 "NOT priced by this model; shown so the (6.13) vs (6.14) distinction is explicit"))

## 3. The anchor — a swing collapsed into a call

One exercise day, a one-day window, and `zero_penalty=True` so leftover quota costs nothing.
That is a European call, and the model should reproduce Black-76 at the variance from §2.

This is the only check in the project against an **independent closed form**; everything else
is internal consistency. A tree or forward-fitting regression would show up here first.

In [ ]:
def swing(days_of_quota, start, end, right, strike=None, n_p=None, vol=None):
    """Price a call swing. `right=True` waives the quota (an option, not an obligation)."""
    params = dict(product_type="call_swing", valDate=VAL_DATE,
                  storageStart=start, storageEnd=end,
                  capacity_mwh=days_of_quota * MWH_PER_DAY, daily_max=MWH_PER_DAY,
                  clips_per_day=1, vol=VOL if vol is None else vol, sMR=SMR,
                  n_p_full=N_P if n_p is None else n_p, run_intrinsic=False,
                  discount_rate=DISCOUNT_RATE,
                  strike=STRIKE if strike is None else strike,
                  zero_penalty=bool(right), daily_curve=CURVE)
    model, _ = sm.run_valuation(None, params)
    value = float(model.v[0, model.n_p, model.initial_state])
    used = float(np.abs(np.asarray(model.exp_ex[:model.n_t])).sum())
    return value / (days_of_quota * MWH_PER_DAY), used, model


_df_one = exp(-DISCOUNT_RATE * T_ONE)
_var_one = ou_var(T_ONE)
_strikes = [STRIKE] if SMOKE else [0.6 * F_LEVEL, 0.8 * F_LEVEL, F_LEVEL,
                                   1.2 * F_LEVEL, 1.5 * F_LEVEL]
_rows = []
for _k in _strikes:
    _model_value, _, _m = swing(1, ONE_DATE, ONE_DATE, True, strike=_k, n_p=N_P_ANCHOR)
    _closed = black76(F_LEVEL, _k, _var_one, _df_one)
    _rows.append({"strike": _k, "model (1-day swing)": _model_value,
                  "Black-76": _closed, "difference": _model_value - _closed,
                  "relative": abs(_model_value - _closed) / max(_closed, 1e-12)})
_anchor = pd.DataFrame(_rows)
display(_anchor.style.hide(axis="index").format({
    "strike": "{:.2f}", "model (1-day swing)": "{:.5f}", "Black-76": "{:.5f}",
    "difference": "{:+.5f}", "relative": "{:.2%}"})
    .set_caption(f"European call on {ONE_DATE:%Y-%m-%d}, forward {F_LEVEL:.2f}, "
                 f"total log-variance {_var_one:.5f} — EUR/MWh"))

# The tree's own realised distribution, as a second opinion on the variance.
_p = _m.prob[_m.Dt].sum(axis=1); _p = _p / _p.sum()
_x = np.asarray(_m.x)[_m.Dt]
_mean = float(np.dot(_p, _x))
_var_tree = float(np.dot(_p, (_x - _mean) ** 2))
print(f"tree's realised log-variance on the expiry day: {_var_tree:.6f}  "
      f"({_var_tree / _var_one - 1:+.2%} against the analytic OU value)")

_worst = _anchor["relative"].max()
if _worst > 0.02:
    raise AssertionError(
        f"the 1-day swing has drifted {_worst:.2%} from Black-76. Either the tree is too "
        f"narrow (N_P_ANCHOR = {N_P_ANCHOR}) or something in the price process has changed — "
        f"do not read the ladder below until this closes.")
print(f"worst deviation from the closed form: {_worst:.2%}")

## 4. The ladder

Now add the differences back one at a time, each row changing exactly one thing from the row
above. Everything is quoted **per MWh of quota**, so the contracts are directly comparable.

| step | what changes |
|---|---|
| European call | one date, one exercise, a right |
| any-day right | the same right, exercisable on any day of the window — the timing option |
| N-day right | N exercises instead of one, still optional |
| N-day obligation | the quota must be met. **This is the model's default** |

In [ ]:
_rows = []
_var_mid = ou_var(T_ONE)
_rows.append({"contract": f"European call, one date ({ONE_DATE:%d %b %Y})",
              "days": 1, "right?": "right", "EUR/MWh": black76(F_LEVEL, STRIKE, _var_mid, _df_one),
              "MWh used": np.nan, "of quota": np.nan})
_v, _u, _ = swing(1, ONE_DATE, ONE_DATE, True, n_p=N_P_ANCHOR)
_rows.append({"contract": "  the same, priced by the model", "days": 1, "right?": "right",
              "EUR/MWh": _v, "MWh used": _u, "of quota": _u / MWH_PER_DAY})
for _d in DAY_COUNTS:
    for _right in (True, False):
        _v, _u, _ = swing(_d, WIN_START, WIN_END, _right)
        _label = ("any day in the window" if _d == 1 else f"{_d} days out of {WINDOW_DAYS}")
        _rows.append({"contract": f"{'RIGHT' if _right else 'OBLIGATION'} to sell {_label}",
                      "days": _d, "right?": "right" if _right else "obligation",
                      "EUR/MWh": _v, "MWh used": _u,
                      "of quota": _u / (_d * MWH_PER_DAY)})
_ladder = pd.DataFrame(_rows)
display(_ladder.style.hide(axis="index").format({
    "days": "{:.0f}", "EUR/MWh": "{:.4f}", "MWh used": "{:,.0f}", "of quota": "{:.1%}"},
    na_rep="—")
    .set_caption(f"call swing vs vanilla call — strike {STRIKE:.2f} on a flat {F_LEVEL:.2f} "
                 f"curve, sVol {VOL}, sMR {SMR}, per MWh of quota, undiscounted"))

_plot = _ladder[~_ladder["contract"].str.startswith("  ")]
fig, ax = plt.subplots(figsize=(12, 0.55 * len(_plot) + 1.2))
_colours = ["tab:grey" if i == 0 else ("tab:green" if r == "right" else "tab:red")
            for i, r in enumerate(_plot["right?"])]
ax.barh(range(len(_plot)), _plot["EUR/MWh"], color=_colours)
for i, v in enumerate(_plot["EUR/MWh"]):
    ax.text(v, i, f" {v:.3f}", va="center", fontsize=8)
ax.set_yticks(range(len(_plot)))
ax.set_yticklabels(_plot["contract"], fontsize=8)
ax.invert_yaxis(); ax.set_xlabel("EUR/MWh of quota")
ax.set_title("Grey: vanilla call.  Green: a right.  Red: an obligation.")
plt.tight_layout(); plt.show()

_call = float(_ladder.iloc[0]["EUR/MWh"])
_anyday = float(_ladder[_ladder["contract"].str.startswith("RIGHT")].iloc[0]["EUR/MWh"])
print(f"timing alone is worth {_anyday/_call - 1:+.0%} over a single-date call "
      f"({_call:.3f} -> {_anyday:.3f} EUR/MWh)")
for _d in DAY_COUNTS:
    _r = float(_ladder[(_ladder["days"] == _d) & (_ladder["right?"] == "right")].iloc[-1]["EUR/MWh"])
    _o = float(_ladder[(_ladder["days"] == _d) & (_ladder["right?"] == "obligation")].iloc[-1]["EUR/MWh"])
    print(f"at {_d:>3} days the obligation costs {1 - _o/_r:.0%} of the right "
          f"({_r:.3f} -> {_o:.3f} EUR/MWh)")

## Traps

- **The vol.** `sVol` is the instantaneous rate. Compare a mean-reverting model to a vanilla
  formula at that number and you will conclude the model is cheap. Use `ou_var(T)`.
- **The obligation.** `call_swing` defaults to a mandatory quota. Benchmarking it against a
  broker's call quote compares two different contracts; set `zero_penalty=True` first.
- **Per MWh of what.** The ladder divides by quota, not by MWh exercised, because that is what
  makes a 1-day and a 90-day contract comparable. A right uses only part of its quota — the
  `of quota` column — so its value per MWh *exercised* is much higher.
- **The strike is the same, the moneyness is not.** On a flat curve every exercise day is
  equally in the money, so this notebook isolates structure. On a shaped curve the day
  selection and the option value mix, which is what `Products.ipynb` is for.
- **Undiscounted.** Both legs would carry the same discount factor here, so it cancels. It
  does not cancel once the contracts differ in *when* they settle — see `Products.ipynb` §6b.